In [3]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=puPHfbhqqCrSuOcksSDCDeow7gZFPh&access_type=offline&code_challenge=LqsSIrxnqvpsZ0N81yDw4sjWxP2h3Lablv4crN7p5ss&code_challenge_method=S256


Credentials saved to file: [/Users/meghakaladharreddypothamsetty/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "zprocure" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [8]:
import os
import base64
from google import genai
from google.genai import types

# --- Gemini Setup ---
client = genai.Client(
    vertexai=True,
    project="aistimate",
    location="global",
)

model_name = "gemini-2.5-pro"

generate_content_config = types.GenerateContentConfig(
    temperature=0,
    top_p=1,
    seed=7,
    max_output_tokens=65535,
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
    ],
    thinking_config=types.ThinkingConfig(thinking_budget=-1),
)

# --- MIME-aware binary loader for any file ---
def make_part(path: str) -> types.Part:
    with open(path, "rb") as f:
        data = f.read()
    ext = os.path.splitext(path)[1].lower()
    mime = {
        ".pdf": "application/pdf",
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".gif": "image/gif",
        ".bmp": "image/bmp",
        ".webp": "image/webp",
        ".txt": "text/plain",
        ".json": "application/json",
    }.get(ext, "application/octet-stream")
    return types.Part.from_bytes(data=data, mime_type=mime)

# --- Your full forensic protocol (as provided) ---
protocol_text = """[START OF PROTOCOL] 
ROLE AND OBJECTIVE: 
You are "AiEstimate Analyst," an AI expert specializing in generating plaintiff-style property loss estimates and performing forensic cost analysis. ... [TRUNCATED FOR SPACE] ...
[END OF PROTOCOL]"""  # Paste your full prompt here without truncating

# --- Stage 2: Generate AiEstimate + Forensic Rebuttal ---
def run_stage2(master_input_path: str) -> str:
    contents = [
        types.Content(role="user", parts=[types.Part.from_text(text=protocol_text)]),
        types.Content(role="model", parts=[types.Part.from_text(text="AiEstimate Forensic Protocol initiated. Standing by for MASTER INPUT DATA.")]),
        types.Content(role="user", parts=[make_part(master_input_path)])
    ]

    print("\n🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...\n")
    stage2_output = ""
    for chunk in client.models.generate_content_stream(
        model=model_name,
        contents=contents,
        config=generate_content_config,
    ):
        stage2_output += chunk.text
    return stage2_output


# --- Stage 3: Compare with Final Estimate ---
def run_stage3(stage2_output: str, master_input_path: str, final_estimate_path: str):
    contents = [
        types.Content(role="user", parts=[types.Part.from_text(text=protocol_text)]),
        types.Content(role="model", parts=[types.Part.from_text(text="AiEstimate Forensic Protocol initiated. Standing by for MASTER INPUT DATA.")]),
        types.Content(role="user", parts=[make_part(master_input_path)]),
        types.Content(role="model", parts=[types.Part.from_text(text=stage2_output)]),
        types.Content(role="user", parts=[make_part(final_estimate_path)]),
    ]
    stage3_output = ""
    print("\n🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...\n")
    for chunk in client.models.generate_content_stream(
        model=model_name,
        contents=contents,
        config=generate_content_config,
    ):
        stage3_output += chunk.text
    return stage3_output


In [9]:
# Step 1: Run Stage 2 with just the carrier estimate
stage2_output = run_stage2("Aiestimate/windstorm and hail - 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Carrier Estimate ($16,113.56) Insurance Carrier Estimate.pdf")

# Optional: save the output for later use
with open("stage2_output.txt", "w") as f:
    f.write(stage2_output)
# Step 2: Later, run Stage 3 with the final estimate for comparison
# (Re-load if needed: stage2_output = open("stage2_output.txt").read())

stage3_output = run_stage3(stage2_output, "Aiestimate/windstorm and hail - 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Carrier Estimate ($16,113.56) Insurance Carrier Estimate.pdf", "Aiestimate/windstorm and hail - 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Estimate - Grace Forensic ($64,191.29) Plaintiff Expert Estimate.pdf")

with open("stage3_output.txt", "w") as f:
    f.write(stage3_output)



🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...

